In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
%load_ext autoreload
%autoreload 2

from src.data import load_train
from src.config import FAMILIES, SEED
from src.evaluation import get_cv, multiclass_metrics, binary_view, evaluate_cv, save_results
from src.models import get_models

X_train, y_train, meta_train = load_train()
print(f"X_train: {X_train.shape}  | families: {dict(y_train.value_counts())}")

In [ ]:
#            0        1        2      3      4        5       6        7      8        9
y_true = ['Normal','Normal','Normal','DoS','DoS','Probe','R2L','U2R','Normal','DoS']
y_pred = ['Normal','Normal','DoS',  'DoS','Normal','Probe','Normal','U2R','Normal','DoS']

# Hand computation:
#   Actual Normal = {0,1,2,8} (4); actual attacks = {3,4,5,6,7,9} (6)
#   FPR = Normal->attack / Normal = 1/4 = 0.25   (row 2)
#   FNR = attack->Normal / attacks = 2/6 = 1/3   (rows 4,6)
#   Per-class F1: Normal=2/3, DoS=2/3, Probe=1, R2L=0, U2R=1  -> macro = 2/3
m = multiclass_metrics(y_true, y_pred, labels=FAMILIES)
b = binary_view(y_true, y_pred)

assert np.isclose(m["macro_f1"], 2/3), f"macro_f1 {m['macro_f1']}"
assert np.isclose(b["fpr"], 0.25), f"fpr {b['fpr']}"
assert np.isclose(b["fnr"], 1/3), f"fnr {b['fnr']}"

# Cross-check macro-F1 against sklearn directly
from sklearn.metrics import f1_score
sk = f1_score(y_true, y_pred, average="macro", labels=FAMILIES, zero_division=0)
assert np.isclose(m["macro_f1"], sk)
print("Test 1: hand-computed macro-F1/FPR/FNR match")

In [ ]:
y5 = ["Normal", "DoS", "Probe", "R2L", "U2R"]

# Perfect predictions
mp = multiclass_metrics(y5, y5, labels=FAMILIES)
bp = binary_view(y5, y5)
assert np.isclose(mp["macro_f1"], 1.0) and np.isclose(bp["fpr"], 0.0) and np.isclose(bp["fnr"], 0.0)

# All-Normal predictions -> FPR 0, FNR 1
bn = binary_view(y5, ["Normal"] * 5)
assert np.isclose(bn["fpr"], 0.0) and np.isclose(bn["fnr"], 1.0)

# All-attack predictions -> FPR 1
ba = binary_view(y5, ["DoS"] * 5)
assert np.isclose(ba["fpr"], 1.0)
print("Test 2: perfect / all-Normal / all-attack edge cases pass")

In [ ]:
cv1, cv2 = get_cv(), get_cv()
s1 = list(cv1.split(X_train, y_train))
s2 = list(cv2.split(X_train, y_train))

for (tr1, va1), (tr2, va2) in zip(s1, s2):
    assert np.array_equal(tr1, tr2) and np.array_equal(va1, va2), "CV not reproducible"

for tr, va in s1:
    assert len(set(tr) & set(va)) == 0, "train/val overlap within a fold"

all_val = np.concatenate([va for _, va in s1])
assert len(all_val) == len(X_train) and len(set(all_val)) == len(X_train), \
    "OOF coverage broken"
print("Test 3: get_cv reproducible; folds disjoint; every row in exactly one val fold")

In [ ]:
models = get_models()
cv = get_cv()

for name, pipe in models.items():
    fold_df, oof = evaluate_cv(pipe, X_train, y_train, cv)

    print(f"\n=== {name} ===")
    print(f"Accuracy : {fold_df['accuracy'].mean():.4f}   (expected ~0.60)")
    print(f"Macro-F1 : {fold_df['macro_f1'].mean():.4f}   (expected ~0.15)")
    print(f"FPR      : {fold_df['fpr'].mean():.4f}   (expected 0.0)")
    print(f"FNR      : {fold_df['fnr'].mean():.4f}   (expected 1.0)")
    for fam in ["DoS", "Probe", "R2L", "U2R"]:
        print(f"  recall_{fam:<5}: {fold_df[f'recall_{fam}'].mean():.4f}   (expected 0.0)")

    path = save_results(name, fold_df)
    print(f"saved -> {path}")

In [ ]:
fold_df, oof = evaluate_cv(models["M0_dummy"], X_train, y_train, get_cv())

assert len(oof) == len(X_train), "OOF length mismatch"
assert set(oof) <= set(FAMILIES), "OOF contains unexpected labels"
assert set(oof) == {"Normal"}, "Dummy(most_frequent) should predict only Normal"

assert np.isclose(fold_df["fpr"].mean(), 0.0)
assert np.isclose(fold_df["fnr"].mean(), 1.0)
for fam in ["DoS", "Probe", "R2L", "U2R"]:
    assert np.isclose(fold_df[f"recall_{fam}"].mean(), 0.0)
print("Dummy behaves exactly as theory predicts — harness is trustworthy")